# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [ ]:
# Import the OpenAI client from the openai library.
from openai import OpenAI

# Import Python's built-in operating system module.
import os

# Import two functions for loading environment variables.
from dotenv import load_dotenv, find_dotenv

# Find the .env file and load its variables.
# usecwd=True makes this work when the notebook is opened from the lab folder.
_ = load_dotenv(find_dotenv(usecwd=True))

# Retrieve the API key from the environment.
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY was not found. Add it to .env or your environment before running the chatbot.")

In [ ]:
# Install Panel into the Python environment used by this notebook.
%pip install panel

What does each component mean?
Component	                        Explanation
from openai                         import OpenAI	Imports the class used to create an OpenAI API client.

import os	                        Gives Python access to operating-system functionality, including environment variables.

dotenv	                            A Python package that can read variables from a .env file.
find_dotenv()	                    Searches for a .env file.

load_dotenv()	                    Loads variables from that file into the environment, without overriding existing values by default.

os.getenv()	                        Reads a variable from the environment. Returns None if it is missing.


A .env file can contain a line such as OPENAI_API_KEY=your_actual_key_here. It keeps credentials separate from your program, but it is not encryption. Never commit your .env or API key to GitHub.

In [ ]:
# Re-run the setup defensively if this cell is executed out of order.
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv(usecwd=True))
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY was not found. Add it to .env or your environment before running the chatbot.")

client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content


def get_completion_from_messages(messages, model="gpt-3.5-turbo", temperature=0): 
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content


In [ ]:
messages =  [  
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},    
{'role':'user', 'content':'tell me a joke'},   
{'role':'assistant', 'content':'Why did the chicken cross the road'},   
{'role':'user', 'content':'I don\'t know'}  ]

In [ ]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

In [ ]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

In [ ]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

In [ ]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

In [ ]:
# Start with the chatbot's initial instructions.
context = [
    {"role": "system", "content": "You are a pizza ordering assistant."}
]

# Add the customer's first message to the history.
context.append({
    "role": "user",
    "content": "I would like a pepperoni pizza."
})

# Add an example assistant response.
context.append({
    "role": "assistant",
    "content": "What size would you like?"
})

# Display the complete conversation history.
print(context)

# Count the messages currently stored.
print("Number of messages:", len(context))

# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant. 

In [ ]:
def collect_messages(_):
    prompt = inp.value
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context) 
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600))
    )
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'}))
    )
 
    return pn.Column(*panels)


In [ ]:
# Install the Jupyter integration for Bokeh-based widgets.
%pip install jupyter_bokeh

In [ ]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

In [ ]:
# Import the Panel library.
import panel as pn

# Activate Panel's Jupyter integration.
pn.extension()

# Create a simple test widget.
test_widget = pn.widgets.TextInput(
    name="Panel Test",
    placeholder="Type something here..."
)

# Display the widget in the notebook.
test_widget

In [ ]:
messages =  context.copy()
messages.append(
    {
        'role': 'system',
        'content': (
            'Create a JSON summary of the previous food order. '
            'Itemize the price for each item. '
            'The fields should be: '
            '1) pizza, include size; '
            '2) list of toppings; '
            '3) list of drinks, include size; '
            '4) list of sides include size; '
            '5) total price.'
        )
    }
)

response = get_completion_from_messages(messages, temperature=0)
print(response)


## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [ ]:
# VERSION 1 — Basic OrderBot
# Define the chatbot's basic instructions.
context_v1 = [
    {
        "role": "system",
        "content": (
            "You are a friendly pizza ordering assistant. "
            "Help customers choose pizzas, toppings, and drinks. "
            "Ask questions when necessary."
        )
    },
    {
        "role": "user",
        "content": "I want a large pepperoni pizza with mushrooms."
    }
]

# Send the messages to the model.
response_v1 = get_completion_from_messages(
    context_v1,
    temperature=0
)

# Display the model's response.
print(response_v1)

In [ ]:
# VERSION 2 — Guided OrderBot
# Add a fixed menu and require missing details instead of inventing them.
context_v2 = [
    {"role": "system", "content": "You are a friendly pizza restaurant assistant.\n"
     "Menu: small/medium/large margherita or pepperoni pizzas; mushrooms and olives;\n"
     "water or cola in small/large sizes.\n"
     "Ask for missing size or drink details. Do not invent prices or unavailable items.\n"
     "Summarize the complete order and ask for confirmation before finalizing it."},
    {"role": "user", "content": "I want a pepperoni pizza and a cola."}
]
response_v2 = get_completion_from_messages(context_v2, temperature=0)
print(response_v2)

In [ ]:
# VERSION 3 — Restricted OrderBot with delivery constraints
# Add pickup/delivery logic and explicitly reject unsupported requests.
context_v3 = [
    {"role": "system", "content": "You are OrderBot for a pizza restaurant.\n"
     "Accept only margherita or pepperoni pizzas, mushrooms or olives, and water or cola.\n"
     "Ask for size, quantity, and pickup or delivery. For delivery, ask for the address.\n"
     "Reject unavailable requests and suggest an available alternative.\n"
     "Confirm the final order before submitting it."},
    {"role": "user", "content": "Can I get a large pepperoni with olives for delivery?"}
]
response_v3 = get_completion_from_messages(context_v3, temperature=0)
print(response_v3)

## Exercise report

I tested three OrderBot prompt versions. Version 1 used only a role and a broad goal,
so it could ask useful follow-up questions but had no menu constraints. Version 2
added an explicit menu and rules for missing details; this made the response more
consistent and reduced invented items or prices. Version 3 added delivery and
confirmation rules and instructed the assistant to reject unavailable requests.

The comparison shows that clearer system instructions improve reliability, but the
model still needs conversation history and application-side validation for a real
ordering system. I learned that prompting controls behavior, while Python code
controls the conversation state and ultimately decides which actions are allowed.